# Lección 9.7 — Inicialización de pesos y gradientes que desaparecen o explotan: el teléfono descompuesto

> **Módulo 9 · Redes neuronales desde la intuición** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 9.3 (funciones de activación), Lección 9.4 (retropropagación y bucle de entrenamiento), Lección 9.5 (optimizadores)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar por qué **el punto de partida** de los pesos decide si una red profunda puede aprender o no.
- Reconocer qué pasa si todos los pesos empiezan **iguales**, **demasiado pequeños** o **demasiado grandes**.
- Elegir la inicialización adecuada: **Xavier/Glorot** (tanh, sigmoide) o **He/Kaiming** (ReLU).
- Diagnosticar **gradientes que desaparecen** y **gradientes que explotan**, y aplicar el *gradient clipping*.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas y un conjunto de datos pequeño: **dígitos escritos a mano** de 8×8 píxeles que vienen incluidos en scikit-learn (no hay que descargar nada).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons, load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
rng = np.random.default_rng(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (8, 4.5)

# Dígitos 8x8 → 64 números entre 0 y 1 por imagen, 10 clases (0–9)
X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X / 16.0, y, test_size=0.25, random_state=0)
X_tr, X_te = torch.tensor(X_tr, dtype=torch.float32), torch.tensor(X_te, dtype=torch.float32)
y_tr, y_te = torch.tensor(y_tr), torch.tensor(y_te)
print(f"✅ Todo listo · {len(X_tr)} imágenes de entrenamiento, {len(X_te)} de prueba · dispositivo: {device}")

---
## 1. 📖 La intuición: el teléfono descompuesto

Diez personas en fila se pasan un mensaje al oído.

- Si cada una lo repite **un poco más bajo** que como lo oyó (digamos, a la mitad de volumen), al final de la fila **no se oye nada**.
- Si cada una lo repite **un poco más fuerte** (al doble), al final es un **grito ensordecedor** donde tampoco se entiende nada.
- Solo si cada persona lo repite **al mismo volumen**, el mensaje llega bien.

Una red profunda es esa fila. La señal (los datos) pasa capa por capa hacia adelante, y en el camino de regreso la **culpa del error** (el gradiente, Lección 9.4) pasa capa por capa hacia atrás. En cada capa se **multiplica** por los pesos. Si esos pesos están mal escalados, la señal se apaga o explota, y la red **no puede aprender**.

> 🎯 Por eso los pesos iniciales no se eligen "al azar, de cualquier manera": se eligen al azar **con el tamaño justo** para que el volumen se mantenga.

### ✋ Ejercicio 1 — Calcula el volumen al final de la fila
Una señal empieza con volumen 1 y atraviesa **20 capas**. Calcula (con calculadora o en la celda de abajo) cuánto vale al final si cada capa la multiplica por:
- a) 0.5 · b) 0.9 · c) 1.0 · d) 1.1 · e) 2.0

✍️ *Mis cálculos:*

In [ ]:
for factor in [0.5, 0.9, 1.0, 1.1, 2.0]:     # ✏️ prueba otros factores o cambia el número de capas
    print(f"factor {factor}: tras 20 capas → {factor ** 20:,.6g}")

<details><summary><b>👀 Ver respuesta</b></summary>

- 0.5 → **0.00000095** (prácticamente cero) · 0.9 → **0.12** · 1.0 → **1** · 1.1 → **6.7** · 2.0 → **1,048,576**.

Lo inquietante es el caso b) y d): factores que parecen "casi 1" se convierten en un 12 % o en casi 7 veces el valor original. **Con muchas capas, cualquier pequeño desequilibrio se multiplica.** Y las redes modernas tienen decenas o cientos de capas.
</details>

---
## 2. 📖 El concepto: ¿por dónde empiezan las perillas?

Antes de entrenar, cada peso de la red necesita un **valor inicial**. El descenso de gradiente (Lección 3.3) parte de ahí. Hay tres formas clásicas de hacerlo mal:

| Error | Qué pasa |
|---|---|
| Todos los pesos **iguales** (por ejemplo, todos en 0 o en 0.5) | Todas las neuronas de una capa calculan **lo mismo** y reciben **la misma corrección**: se quedan clonadas para siempre. |
| Pesos **demasiado pequeños** | La señal se apaga capa a capa: las últimas capas reciben casi ceros. |
| Pesos **demasiado grandes** | La señal crece sin control o satura las activaciones (sigmoide y tanh se quedan pegadas en sus extremos). |

Empecemos por el primero, el más sorprendente: **la simetría**.

In [ ]:
X_lunas, y_lunas = make_moons(n_samples=400, noise=0.2, random_state=0)
Xl = torch.tensor(X_lunas, dtype=torch.float32)
yl = torch.tensor(y_lunas, dtype=torch.float32).unsqueeze(1)

def red_lunas(peso_constante=None):
    torch.manual_seed(0)
    red = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1))
    if peso_constante is not None:                 # todas las perillas empiezan con el MISMO valor
        for capa in [red[0], red[2]]:
            nn.init.constant_(capa.weight, peso_constante)
            nn.init.zeros_(capa.bias)
    opt = torch.optim.Adam(red.parameters(), lr=0.05)
    for paso in range(500):                         # el bucle estándar de la Lección 9.4
        opt.zero_grad()
        perdida = nn.functional.binary_cross_entropy_with_logits(red(Xl), yl)
        perdida.backward()
        opt.step()
    exactitud = ((red(Xl) > 0).float() == yl).float().mean().item()
    return red, exactitud

### 🔮 Ejercicio 2 — Ocho neuronas clonadas
Vamos a entrenar la misma red (8 neuronas ocultas) dos veces:
1. con pesos iniciales **aleatorios** (lo normal);
2. con **todos** los pesos iniciales iguales a 0.5.

Predice: en el caso 2, después de entrenar, ¿las 8 neuronas ocultas tendrán pesos distintos o iguales? ¿Qué exactitud esperas comparada con el caso 1?

✍️ *Mi predicción:*

In [ ]:
red_azar, acc_azar = red_lunas(peso_constante=None)
red_igual, acc_igual = red_lunas(peso_constante=0.5)     # ✏️ prueba 0.0

print(f"Exactitud con pesos aleatorios : {acc_azar:.0%}")
print(f"Exactitud con pesos todos 0.5  : {acc_igual:.0%}\n")
print("Pesos de las 8 neuronas ocultas tras entrenar (inicio igual):")
print(red_igual[0].weight.data.numpy().round(3))

<details><summary><b>👀 Ver respuesta</b></summary>

Con pesos aleatorios la red llega a ≈97 %. Con todos los pesos en 0.5, las **8 filas son idénticas**: todas las neuronas empezaron igual, recibieron exactamente la misma culpa en cada paso y, por tanto, se movieron igual. Es como tener **una sola neurona copiada 8 veces**, y la exactitud se queda en ≈88 %, lo que logra una frontera casi recta.

Si pruebas `peso_constante=0.0` es aún peor: con todo en cero, la culpa que llega a la capa oculta es cero y esas neuronas **ni siquiera se mueven**.

👉 Regla: **la inicialización aleatoria sirve para romper la simetría.** Cada neurona necesita empezar distinta para especializarse en algo distinto.
</details>

---
## 3. 📖 Cómo funciona por dentro: el volumen de la señal capa a capa

Ya sabemos que hay que empezar **al azar**. Pero ¿al azar de qué tamaño? Hagamos el experimento del teléfono descompuesto con una red de verdad, **a mano con numpy**:

- 1,000 ejemplos con 256 características pasan por **20 capas** de 256 neuronas.
- Los pesos son números aleatorios multiplicados por una **escala** que elegimos.
- En cada capa medimos la **desviación estándar** de las activaciones: el "volumen" de la señal. Si se va a 0, la señal murió.

In [ ]:
def volumen_por_capa(escala, activacion, n_capas=20, ancho=256):
    r = np.random.default_rng(1)
    a = np.random.default_rng(0).normal(size=(1000, ancho))   # la señal de entrada (volumen ≈ 1)
    volumenes, ultima = [], None
    for _ in range(n_capas):
        W = r.normal(size=(ancho, ancho)) * escala      # pesos aleatorios de tamaño "escala"
        a = activacion(a @ W)                           # suma ponderada + activación
        volumenes.append(a.std())
    return volumenes, a

tanh = np.tanh
relu = lambda z: np.maximum(0, z)

def comparar_escalas(casos, activacion, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for nombre, escala in casos.items():
        vols, ultima = volumen_por_capa(escala, activacion)
        axes[0].plot(range(1, 21), vols, "o-", label=nombre)
        axes[1].hist(ultima.ravel(), bins=60, alpha=.5, label=nombre)
        print(f"{nombre:<22} volumen en capa 1: {vols[0]:.3g}   capa 10: {vols[9]:.3g}   capa 20: {vols[19]:.3g}")
    axes[0].set_yscale("log"); axes[0].set_xlabel("capa"); axes[0].set_ylabel("volumen (desv. estándar)")
    axes[0].set_title(titulo); axes[0].legend(); axes[1].set_title("Activaciones en la capa 20"); axes[1].legend()
    plt.tight_layout(); plt.show()

### 🔮 Ejercicio 3 — ¿Muy bajito o muy fuerte?
Usaremos **tanh** (Lección 9.3), que siempre devuelve valores entre −1 y 1. Probaremos dos escalas:
- **0.01** (pesos diminutos)
- **1.0** (pesos "normales" de una distribución estándar)

Predice para cada una: ¿qué pasa con el volumen al llegar a la capa 20? Pista para la escala 1.0: cada neurona suma **256** entradas.

✍️ *Mi predicción:*

In [ ]:
comparar_escalas({"escala 0.01": 0.01, "escala 1.0": 1.0}, tanh, "tanh: dos escalas mal elegidas")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Escala 0.01:** el volumen cae de ≈0.16 en la capa 1 a ≈0.0001 en la capa 5 y a ≈10⁻¹⁶ en la capa 20. La señal **murió**: todas las activaciones son prácticamente 0 y las últimas capas no reciben información de la entrada.
- **Escala 1.0:** el volumen se queda en ≈0.97, que *parece* sano… pero mira el histograma: casi todas las activaciones están pegadas en **−1 o +1**. Al sumar 256 entradas con pesos grandes, las sumas son enormes y tanh **se satura**. Y en la zona saturada la pendiente de tanh es casi 0 (Lección 9.3), así que el gradiente no pasa.

Los dos casos son malos, por razones opuestas: uno apaga la señal, el otro la "clava" en los extremos.
</details>

### 📖 La solución: escalar según cuántas entradas tiene cada neurona

Si una neurona suma `n` entradas, la suma crece con `n`. Para compensarlo, los pesos se hacen más pequeños cuando hay más entradas. Dos recetas famosas:

| Inicialización | Escala de los pesos | Pensada para | En PyTorch |
|---|---|---|---|
| **Xavier / Glorot** (2010) | ≈ `1 / √n` | tanh, sigmoide | `nn.init.xavier_normal_` / `xavier_uniform_` |
| **He / Kaiming** (2015) | ≈ `√(2 / n)` | ReLU y familia | `nn.init.kaiming_normal_(…, nonlinearity="relu")` |

¿Por qué He lleva un **2** extra? Porque ReLU **apaga la mitad** de las activaciones (todas las negativas pasan a 0), y eso reduce el volumen a la mitad en cada capa. El 2 lo compensa.

> 📝 `n` es el número de entradas de la capa (en inglés, *fan-in*). Con n = 256: Xavier ≈ 0.0625, He ≈ 0.088.

### 🔮 Ejercicio 4 — Cada activación con su receta
Ahora con **ReLU**. Compararemos Xavier (pensada para tanh) contra He (pensada para ReLU). ¿Cuál mantendrá el volumen hasta la capa 20? ¿Qué le pasará a la otra?

✍️ *Mi predicción:*

In [ ]:
n = 256
comparar_escalas({"Xavier 1/√n": 1 / np.sqrt(n), "He √(2/n)": np.sqrt(2 / n)}, relu,
                 "ReLU: Xavier vs. He")

<details><summary><b>👀 Ver respuesta</b></summary>

- **He** mantiene el volumen en el mismo orden de magnitud: ≈0.82 en la capa 1 y ≈0.65 en la capa 20.
- **Xavier con ReLU** pierde volumen en cada capa (ReLU se come la mitad de la señal y nadie lo compensa): ≈0.58 → ≈0.02 en la capa 10 → ≈0.0006 en la capa 20. Es el caso "0.9 elevado a 20" del Ejercicio 1.

✏️ Si cambias la celda a `tanh` con Xavier verás que se porta mucho mejor que con ReLU. **Cada activación tiene su receta.**
</details>

---
## 4. 📖 ¿Qué pasa si…? Gradientes que desaparecen

Hasta ahora miramos la señal **hacia adelante**. Pero lo que de verdad importa para aprender es la culpa que viaja **hacia atrás** (Lección 9.4): el gradiente de cada capa. Si llega casi en cero a las primeras capas, esas capas **no aprenden nada**.

Este problema se llama **desvanecimiento del gradiente** (*vanishing gradients*) y fue, durante años, la razón por la que nadie lograba entrenar redes profundas. Sus dos culpables clásicos:
1. **Sigmoide**: su pendiente máxima es 0.25 (Lección 9.3). Multiplicar por ≤ 0.25 en cada capa es el teléfono descompuesto a volumen ¼.
2. **Una mala inicialización**, como acabamos de ver.

Midámoslo en una red de **10 capas ocultas** sobre los dígitos.

In [ ]:
def red_profunda(activacion, inicializacion=None, n_capas=10, ancho=64):
    capas, entrada = [], 64
    for _ in range(n_capas):
        lineal = nn.Linear(entrada, ancho)
        if inicializacion == "he":
            nn.init.kaiming_normal_(lineal.weight, nonlinearity="relu"); nn.init.zeros_(lineal.bias)
        elif inicializacion == "xavier":
            nn.init.xavier_normal_(lineal.weight); nn.init.zeros_(lineal.bias)
        capas += [lineal, activacion()]
        entrada = ancho
    capas.append(nn.Linear(entrada, 10))
    return nn.Sequential(*capas)

def normas_gradiente(red):
    # Un solo paso hacia atrás: ¿cuánta culpa le llega a cada capa?
    red.zero_grad()
    nn.functional.cross_entropy(red(X_tr), y_tr).backward()
    return [capa.weight.grad.norm().item() for capa in red if isinstance(capa, nn.Linear)]

### 🔮 Ejercicio 5 — ¿Cuánta culpa llega a la primera capa?
Compararemos una red **sigmoide** (con la inicialización por defecto de PyTorch) contra una red **ReLU + He**. Para cada una mediremos el tamaño del gradiente en las 11 capas lineales (de la 1, junto a la entrada, a la 11, la salida).

En la red sigmoide, ¿cuántas veces más pequeño crees que será el gradiente de la capa 1 que el de la capa 11? ¿10 veces? ¿1,000? ¿Más?

✍️ *Mi apuesta:*

In [ ]:
torch.manual_seed(0); g_sigmoide = normas_gradiente(red_profunda(nn.Sigmoid))
torch.manual_seed(0); g_relu_he = normas_gradiente(red_profunda(nn.ReLU, "he"))

plt.semilogy(range(1, 12), g_sigmoide, "o-", label="sigmoide (init por defecto)")
plt.semilogy(range(1, 12), g_relu_he, "s-", label="ReLU + He")
plt.xlabel("capa (1 = junto a la entrada)"); plt.ylabel("tamaño del gradiente (escala log)")
plt.title("¿Cuánta culpa le llega a cada capa?"); plt.legend(); plt.grid(alpha=.3); plt.show()

print(f"Sigmoide : capa 1 = {g_sigmoide[0]:.1e}   capa 11 = {g_sigmoide[-1]:.1e}   → {g_sigmoide[-1] / g_sigmoide[0]:,.0f} veces más pequeño")
print(f"ReLU + He: capa 1 = {g_relu_he[0]:.1e}   capa 11 = {g_relu_he[-1]:.1e}")

<details><summary><b>👀 Ver respuesta</b></summary>

¡Unas **200 millones de veces** más pequeño (≈10⁻⁹ contra ≈0.25)! El gradiente se encoge más o menos por 7 en cada capa sigmoide. En la gráfica (escala logarítmica) es una rampa recta: cada capa pierde la misma proporción, como en el teléfono descompuesto.

Con **ReLU + He** el gradiente es de tamaño parecido (≈0.1–0.2) en **todas** las capas: la culpa llega bien a todos.
</details>

### 🔮 Ejercicio 6 — ¿Y eso importa al entrenar?
Ahora entrenamos las dos redes 30 épocas con **Adam** (Lección 9.5), el optimizador que suele "salvar" situaciones difíciles. Predice la exactitud en prueba de cada una.

✍️ *Mi predicción:* sigmoide → ___ % · ReLU + He → ___ %

In [ ]:
def entrenar(red, epocas=30, lr=1e-3, optimizador="adam", recorte=None):
    opt = (torch.optim.Adam(red.parameters(), lr=lr) if optimizador == "adam"
           else torch.optim.SGD(red.parameters(), lr=lr))
    historia = []
    for _ in range(epocas):
        orden = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), 64):                     # mini-lotes de 64 (Lección 3.5)
            idx = orden[i:i + 64]
            opt.zero_grad()
            perdida = nn.functional.cross_entropy(red(X_tr[idx]), y_tr[idx])
            perdida.backward()
            if recorte is not None:                           # gradient clipping (sección 5)
                nn.utils.clip_grad_norm_(red.parameters(), max_norm=recorte)
            opt.step()
        with torch.no_grad():
            historia.append(nn.functional.cross_entropy(red(X_tr), y_tr).item())
    exactitud = (red(X_te).argmax(1) == y_te).float().mean().item()
    return historia, exactitud

torch.manual_seed(0); h_sig, acc_sig = entrenar(red_profunda(nn.Sigmoid))
torch.manual_seed(0); h_relu, acc_relu = entrenar(red_profunda(nn.ReLU, "he"))

plt.plot(h_sig, label=f"sigmoide · prueba {acc_sig:.0%}")
plt.plot(h_relu, label=f"ReLU + He · prueba {acc_relu:.0%}")
plt.axhline(np.log(10), ls="--", c="gray", label="ln(10) ≈ 2.30: adivinar al azar")
plt.xlabel("época"); plt.ylabel("pérdida de entrenamiento"); plt.legend(); plt.show()
print(f"Sigmoide : pérdida final {h_sig[-1]:.3f} · exactitud en prueba {acc_sig:.0%}")
print(f"ReLU + He: pérdida final {h_relu[-1]:.3f} · exactitud en prueba {acc_relu:.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Sigmoide:** la pérdida se queda **clavada en ≈2.30** durante las 30 épocas y la exactitud es ≈8 %: peor que adivinar al azar entre 10 dígitos. Ni Adam la salva: las primeras capas no reciben señal y lo que llega a la salida es "ruido sin información".
- **ReLU + He:** la pérdida baja casi a 0 y la exactitud en prueba es ≈97 %.

La línea gris es una pista de diagnóstico que usarás mucho (Lección 9.9): con 10 clases, una red que **no sabe nada** tiene pérdida ≈ ln(10) ≈ 2.30. Si tu pérdida se queda ahí, tu red no está aprendiendo.
</details>

---
## 5. 📖 ¿Qué pasa si…? Gradientes que explotan

El problema contrario: pesos un poco **demasiado grandes** en una red profunda ReLU. Cada capa amplifica la señal, la pérdida inicial es enorme, el gradiente también, y un solo paso del optimizador lanza los pesos a valores absurdos. En pocos pasos aparecen números infinitos y la pérdida se vuelve **`nan`** (*not a number*, "no es un número").

La herramienta de emergencia se llama **recorte de gradiente** (*gradient clipping*): si el tamaño total del gradiente supera un límite (`max_norm`), se encoge para que lo iguale. La **dirección** se conserva; solo se limita la **longitud del paso**. Es como el limitador de velocidad de un coche.

```python
perdida.backward()
nn.utils.clip_grad_norm_(red.parameters(), max_norm=1.0)   # ← entre backward() y step()
opt.step()
```

In [ ]:
def red_grande(desviacion, n_capas=10):
    # Red ReLU de 10 capas cuyos pesos iniciales tienen la desviación estándar que elijamos
    torch.manual_seed(0)
    red = red_profunda(nn.ReLU, n_capas=n_capas)
    for capa in red:
        if isinstance(capa, nn.Linear):
            nn.init.normal_(capa.weight, 0, desviacion); nn.init.zeros_(capa.bias)
    return red

print(f"Desviación que recomienda He para 64 entradas: √(2/64) = {np.sqrt(2 / 64):.3f}")

### 🔮 Ejercicio 7 — Pesos un poco grandes
Vamos a usar una desviación de **0.3** (He recomienda ≈0.18) y entrenar con **SGD**, `lr=0.01`, sin recorte y con recorte (`max_norm=1`). Predice qué saldrá en cada caso: ¿aprende, no aprende o `nan`?

✍️ *Mi predicción:*

In [ ]:
DESVIACION = 0.3      # ✏️ prueba 0.18 (He) y 0.4

for recorte in [None, 1.0]:
    torch.manual_seed(0)
    historia, acc = entrenar(red_grande(DESVIACION), epocas=20, lr=0.01, optimizador="sgd", recorte=recorte)
    print(f"recorte={str(recorte):<5} pérdida época 1: {historia[0]:>8.3g}   época 20: {historia[-1]:>8.3g}   exactitud prueba: {acc:.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Sin recorte:** la pérdida es `nan` desde la primera época y la exactitud ≈8 % (la red predice basura). El gradiente explotó: en algún paso midió miles de millones y lanzó los pesos al infinito.
- **Con recorte:** la pérdida empieza altísima (≈77 tras la primera época) pero **baja** (≈2.7 en la época 20) y la exactitud llega a ≈50 %. El recorte evitó el desastre, pero la red sigue arrastrando un mal comienzo.
- ✏️ Con **0.18 (He)** y sin recorte la red aprende bien (pérdida ≈0.2, exactitud ≈90 %, sin `nan`): **la verdadera solución era inicializar bien**. Y con 0.4 ni el recorte alcanza: la pérdida empieza en miles y la exactitud no pasa de ≈46 %. El recorte es un **cinturón de seguridad**, no un sustituto.
</details>

---
## 6. 📖 Las perillas de esta lección

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| Escala de la inicialización | Tamaño de los pesos al empezar | Activaciones saturadas o que explotan, pérdida inicial enorme | Señal y gradiente que se apagan | **He** con ReLU, **Xavier** con tanh/sigmoide (o el valor por defecto de PyTorch en redes poco profundas) |
| `nonlinearity` en `kaiming_*` | Ajusta la receta a la activación | — | — | `"relu"` (o `"leaky_relu"`) |
| `max_norm` (recorte) | Longitud máxima del paso del gradiente | Recorta menos (casi sin efecto) | Pasos muy cortos: entrena más lento | 1.0 (redes recurrentes y Transformers: 0.5–5) |
| Profundidad | Número de capas que atraviesa la señal | Más riesgo de desvanecer/explotar | Menos riesgo | Empieza poco profundo y añade capas solo si mejora la validación |

> 💡 La inicialización por defecto de `nn.Linear` en PyTorch es razonable para redes **poco profundas** (2–4 capas, como casi todas las del módulo). En redes profundas con ReLU conviene usar He explícitamente. ✏️ Pruébalo: mide `normas_gradiente(red_profunda(nn.ReLU))` (sin `"he"`) y compárala con la versión He.

### 🔮 Ejercicio 8 — ¿Por qué hoy entrenamos redes de 100 capas?
Si las redes profundas sufren tanto, ¿cómo existen ResNets de 152 capas o Transformers de 96? Además de **ReLU** e **inicializaciones** como He, hay dos inventos que verás pronto. Antes de abrir la respuesta, piensa: ¿qué harías tú para que la culpa llegue a la primera capa sin atravesar 100 multiplicaciones?

✍️ *Mi idea:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Batch normalization** (Lección 9.8): una capa que **re-normaliza** las activaciones en cada paso para que su volumen se mantenga cerca de 1, pase lo que pase con los pesos. Es como un técnico de sonido en cada persona de la fila.
2. **Conexiones residuales** (Lección 10.4, ResNet): un "atajo" que suma la entrada de un bloque a su salida (`salida = bloque(x) + x`). La culpa puede viajar hacia atrás por el atajo **sin multiplicarse**. Es como si el mensaje, además de pasar de oído en oído, viajara también por un cable directo.

Si se te ocurrió "un camino directo" o "reajustar el volumen en cada capa", pensaste como los investigadores que resolvieron el problema.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| La pérdida se queda clavada en ≈ ln(n_clases) desde el principio | Gradientes que desaparecen (sigmoide/tanh profunda, pesos diminutos) | ReLU + He, menos capas, BatchNorm (9.8) |
| Varias neuronas tienen exactamente los mismos pesos | Inicialización constante: no se rompió la simetría | Inicialización aleatoria (nunca todo igual) |
| Pérdida inicial muchísimo mayor que ln(n_clases) | Pesos iniciales demasiado grandes | He/Xavier; revisar escala de la entrada (Lección 2.3) |
| La pérdida se vuelve `nan` o `inf` de golpe | Gradientes que explotan (o tasa de aprendizaje muy alta, Lección 9.6) | `clip_grad_norm_`, bajar `lr`, revisar la inicialización |
| Las primeras capas casi no cambian durante el entrenamiento | Gradiente desvanecido en esas capas | Medir la norma del gradiente por capa (como en la sección 4) |

### 🔮 Ejercicio 9 — Diagnostica
Una compañera te muestra la norma del gradiente de las capas de su red de 8 capas (de la 1 a la 8):

`[3e-07, 2e-06, 1e-05, 9e-05, 6e-04, 4e-03, 3e-02, 2e-01]`

¿Qué problema tiene? ¿Qué activación apostarías a que usa? ¿Qué le recomendarías primero?

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Gradiente que desaparece:** se encoge unas 7 veces por capa, igual que nuestra red sigmoide de la sección 4. Apuesta razonable: **sigmoide** (o tanh con una inicialización pequeña). Primera recomendación: cambiar a **ReLU con inicialización He**. Si necesita muchas capas, añadir **BatchNorm** o conexiones residuales.
</details>

---
## 8. 📋 Ficha: inicialización y salud del gradiente

| | |
|---|---|
| **Qué es** | Elegir el valor inicial de los pesos para que la señal y el gradiente mantengan su "volumen" a través de las capas |
| **Por qué importa** | En una red profunda, cada capa multiplica: un pequeño desequilibrio por capa se vuelve enorme |
| **Recetas** | Xavier/Glorot (≈1/√n) para tanh y sigmoide · He/Kaiming (≈√(2/n)) para ReLU |
| **Señales de que algo falla** | Pérdida clavada en ln(n_clases), normas de gradiente que caen en picada hacia la entrada, `nan` |
| **Herramientas** | `nn.init.*`, `clip_grad_norm_`, ReLU, BatchNorm (9.8), conexiones residuales (10.4) |
| **Regla de oro** | Aleatorio para romper la simetría, con la escala justa para la activación que uses |

## 9. 🧠 Resumen en 5 puntos
1. Una red profunda es un **teléfono descompuesto**: la señal (adelante) y la culpa (atrás) se multiplican en cada capa.
2. Pesos iniciales **iguales** clonan las neuronas: la aleatoriedad sirve para **romper la simetría**.
3. Pesos **demasiado pequeños** apagan la señal; **demasiado grandes** la saturan o la hacen explotar.
4. **Xavier** (tanh/sigmoide) y **He** (ReLU) escalan los pesos según el número de entradas para mantener el volumen.
5. **Gradientes que desaparecen** → pérdida clavada en ln(n_clases); **gradientes que explotan** → `nan`. Remedios: ReLU + He, recorte de gradiente, BatchNorm y conexiones residuales.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** Inicializas **todos** los pesos de una red con el valor 0.1. ¿Qué pasará?
- a) Entrena igual que con pesos aleatorios · b) Las neuronas de cada capa quedan clonadas · c) La pérdida será `nan` · d) Sobreajusta

**2.** ¿Qué inicialización elegirías para una red de 20 capas con ReLU?
- a) Todos los pesos en 0 · b) Xavier · c) He/Kaiming · d) Pesos normales con desviación 1

**3.** En `clip_grad_norm_(parametros, max_norm=1.0)`, ¿qué se modifica?
- a) Los pesos · b) La dirección del gradiente · c) La longitud del gradiente, si supera 1 · d) La tasa de aprendizaje

**4.** Tu red de 10 clases tiene pérdida 2.30 en la época 1 y 2.30 en la época 50. ¿Qué es lo **menos** probable?
- a) Gradientes que desaparecen · b) La red ya aprendió todo lo posible · c) Una red sigmoide demasiado profunda · d) Una tasa de aprendizaje mal elegida

**5.** ¿Por qué la receta He tiene un 2 que Xavier no tiene?
- a) Porque ReLU anula la mitad de las activaciones · b) Porque He es más moderno · c) Para que los pesos sean negativos · d) Porque ReLU satura en 1

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Cualquier valor constante produce simetría (Ejercicio 2), no solo el 0.
2. **c)** He está diseñada para ReLU; Xavier se queda corta (Ejercicio 4) y desviación 1 explota.
3. **c)** Se conserva la dirección, solo se limita la longitud del paso.
4. **b)** 2.30 ≈ ln(10) es la pérdida de quien **adivina al azar**: la red no aprendió nada, no "todo lo posible".
5. **a)** Al anular la mitad, el volumen se divide entre 2 en cada capa; el 2 lo compensa.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Una regresión logística (Lección 5.2) se puede inicializar con todos los pesos en cero sin ningún problema. ¿Por qué ahí no hay "problema de simetría"?

**B.** Un colega dice: *"Si mis gradientes explotan, pongo `max_norm=0.0001` y ya está, nunca más tendré `nan`"*. ¿Qué le responderías?

**C. Caso 💼.** Tu equipo heredó una red de 30 capas con sigmoide que "nunca terminó de entrenar". El jefe propone comprar una GPU más potente para entrenarla más épocas. ¿Qué le dirías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** La logística tiene **una sola neurona**: no hay neuronas "hermanas" que puedan quedar clonadas. La simetría solo es un problema cuando varias neuronas de la misma capa deberían especializarse en cosas distintas.

**B.** Que evitará el `nan`, pero los pasos serán tan cortos que la red **casi no aprenderá**. El recorte es un cinturón de seguridad (valores típicos 0.5–5); el problema de fondo (inicialización, tasa de aprendizaje) hay que arreglarlo aparte.

**C.** Que más épocas no van a ayudar: con 30 capas sigmoide el gradiente que llega a las primeras capas es prácticamente cero (en nuestra red de 10 capas ya era 10⁻⁹). Es un problema de **diseño**, no de potencia. Propón ReLU + He, BatchNorm o una arquitectura con conexiones residuales; probablemente entrene en minutos en la máquina que ya tienen.
</details>

### 10.3 🧩 Práctica integradora: "rescata la red profunda"

Recibiste una red de **8 capas con tanh** que no aprende. La vas a diagnosticar y rescatar en 4 pasos. **No avances sin revisar cada respuesta.**

**Paso 1 ▶️ — El paciente.** Esta es la red tal como te la entregaron: pesos iniciales con desviación **0.01**.

In [ ]:
def red_paciente(desviacion=0.01, inicializacion=None):
    torch.manual_seed(0)
    red = red_profunda(nn.Tanh, n_capas=8)
    for capa in red:
        if isinstance(capa, nn.Linear):
            if inicializacion == "xavier":
                nn.init.xavier_normal_(capa.weight)
            else:
                nn.init.normal_(capa.weight, 0, desviacion)
            nn.init.zeros_(capa.bias)
    return red

torch.manual_seed(0)
historia, acc = entrenar(red_paciente(), epocas=15)
print("Pérdida por época:", [round(h, 2) for h in historia])
print(f"Exactitud en prueba: {acc:.0%}")

✍️ *¿Qué ves?*

<details><summary><b>👀 Ver respuesta</b></summary>

La pérdida se queda **clavada en 2.30 ≈ ln(10)** las primeras épocas; luego se mueve un poco y se vuelve a atascar (≈1.7), y la exactitud en prueba es pésima (≈18 %). Síntoma clásico de señal/gradiente que se apaga: la red pasa la mayor parte del tiempo sin aprender nada.
</details>

**Paso 2 ✋ — La radiografía.** Mide la norma del gradiente por capa del paciente. ¿Confirma tu sospecha?

In [ ]:
normas = normas_gradiente(red_paciente())
for i, g in enumerate(normas, 1):
    print(f"capa {i}: {g:.1e}")

<details><summary><b>👀 Ver respuesta</b></summary>

Sí, y es aún peor de lo esperado: el gradiente es del orden de **10⁻⁹–10⁻¹⁰ en todas las capas, incluida la de salida**. ¿Por qué también la última? Porque la señal **hacia adelante** ya se había apagado (el caso "escala 0.01" de la sección 3): a la capa de salida le llegan entradas casi nulas, así que tampoco puede calcular cuánto corregir sus pesos. Es **desvanecimiento** en las dos direcciones, causado por pesos iniciales diminutos.
</details>

**Paso 3 ✏️ — El tratamiento.** Cambia la inicialización a la receta adecuada para **tanh** y vuelve a entrenar. Completa la línea marcada (pista: ¿qué receta de la tabla de la sección 3 corresponde a tanh?).

In [ ]:
INICIALIZACION = "xavier"   # ✏️ (ya está puesta; prueba también None para comparar)

torch.manual_seed(0)
historia, acc = entrenar(red_paciente(inicializacion=INICIALIZACION), epocas=15)
print(f"Pérdida época 1: {historia[0]:.3f} · época 15: {historia[-1]:.3f} · exactitud en prueba: {acc:.0%}")
print("Normas del gradiente por capa:", [f"{g:.1e}" for g in normas_gradiente(red_paciente(inicializacion=INICIALIZACION))])

<details><summary><b>👀 Ver respuesta</b></summary>

Con **Xavier** (la receta para tanh) la pérdida baja con claridad y la exactitud en prueba sube a ≈96 %. Las normas del gradiente ahora son del mismo orden en todas las capas. **Mismo modelo, mismos datos, mismo optimizador: solo cambió el punto de partida.**
</details>

**Paso 4 ✋ — El informe.** Escribe en tres frases, para alguien que no sabe ML, qué le pasaba a la red y cómo la arreglaste.

✍️ *Mi informe:*

<details><summary><b>👀 Ver un ejemplo</b></summary>

*"La red tenía 8 capas y sus valores iniciales eran tan pequeños que la información se iba apagando al pasar de una capa a otra, como un mensaje susurrado en una fila muy larga. Por eso las primeras capas nunca recibían indicaciones de cómo mejorar y la red adivinaba al azar. Ajustamos el tamaño de esos valores iniciales con una receta estándar (Xavier) y la red pasó de acertar el 10 % a acertar alrededor del 96 % sin cambiar nada más."*
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** deeplearning.ai, *Initializing neural networks* (notas interactivas donde mueves la escala de los pesos y ves las activaciones capa a capa).
- 📄 **Artículos originales:** Glorot y Bengio (2010), *Understanding the difficulty of training deep feedforward neural networks*; He et al. (2015), *Delving Deep into Rectifiers*.
- 📘 **Documentación:** PyTorch, `torch.nn.init` y `torch.nn.utils.clip_grad_norm_`.

---
⬅️ **Anterior:** [9.6 Hiperparámetros de la red: el tablero de perillas](9.6-hiperparametros-de-la-red.ipynb) · ➡️ **Siguiente:** [9.8 Regularización en redes: entrenar para el examen, no para memorizar las respuestas](9.8-regularizacion-en-redes.ipynb)